# Archived 2023 nutrition analysis
Author: Shamseldeen Ismaiil. Original publication: 11 December 2023.

Portfolio archive prepared 23 September 2026. Original code is preserved; stored outputs are removed for portability and one unsupported health-outcome statement is replaced by a descriptive limitation. This historical notebook has not been rerun.

Known issues include missing-value zero imputation, a mismatch between the described and actual final modeling frame, in-sample model comparison, and extrapolation to impossible nutrient combinations. Read README.md and use reviewed_analysis.py for the reproducible 2026 review.

# What is good food?

## 📖 Background
You and your friend have gotten into a debate about nutrition. Your friend follows a high-protein diet and does not eat any carbohydrates (no grains, no fruits). You claim that a balanced diet should contain all nutrients but should be low in calories. Both of you quickly realize that most of what you know about nutrition comes from mainstream and social media.

Being the data scientist that you are, you offer to look at the data yourself to answer a few key questions.

## 💾 The data

You source nutrition data from USDA's FoodData Central [website](https://fdc.nal.usda.gov/download-datasets.html). This data contains the calorie content of 7,793 common foods, as well as their nutritional composition. Each row represents one food item, and nutritional values are based on a 100g serving. Here is a description of the columns:

- **FDC_ID**: A unique identifier for each food item in the database.
- **Item**: The name or description of the food product.
- **Category**: The category or classification of the food item, such as "Baked Products" or "Vegetables and Vegetable Products".
- **Calories**: The energy content of the food, presented in kilocalories (kcal).
- **Protein**: The protein content of the food, measured in grams.
- **Carbohydrate**: The carbohydrate content of the food, measured in grams.
- **Total fat**: The total fat content of the food, measured in grams.
- **Cholesterol**: The cholesterol content of the food, measured in milligrams.
- **Fiber**: The dietary fiber content of the food, measured in grams.
- **Water**: The water content of the food, measured in grams.
- **Alcohol**: The alcohol content of the food (if any), measured in grams.
- **Vitamin C**: The Vitamin C content of the food, measured in milligrams.

In [ ]:
import pandas as pd
df_food = pd.read_csv('nutrition.csv')


## summary:

Create a report that covers the following:

1. fruit has the highest vitamin C and  some other sources of vitamin C.
2. the relationship between the calories and water content 
3. possible drawbacks of a zero-carb diet drawbacks of a very high-protein diet.
4. fit a linear model to find that kcal in protein, carbohydrates and fat.
5. Alcohol as a source of calories.

🥇First of all we need to explore our data and see information about dataframe.
Information of Food data and columns types.

In [ ]:
df_food.info()

Delete all missing Data.
New Data information.

In [ ]:
df_food_Nna = df_food.dropna()
df_food_Nna.info()

All columns are object but we need to convert all columns with data of object types and all data in it to numbers,so we can process this data.

In [ ]:
import numpy as np
df_food[['Vitamin C','Cholesterol']] = df_food[['Vitamin C','Cholesterol']].fillna('0.0 mg')
df_food[['Fiber','Alcohol']]= df_food[['Fiber','Alcohol']].fillna('0.0 g')



We need all data, so we replace missing data with 0, as start to begin aur journey.🌋

In [ ]:
def DataframeAddCol(df, dic):
    """
    Function to convert string columns with 'mg, g, kca, ...' to float and add new columns to the dataframe.
    df: DataFrame, dic: dictionary of column names and measurements like mg, g, ...
    """
    for i, x in dic.items():
        new_col_name = i + "_" + x
        df[new_col_name] = pd.to_numeric(df[i].str.split(x).str[0], errors='coerce')
    return df

Our first step convert all columns with data from objects to numbers.🪜

In [ ]:
dictcol = {"Calories":"kcal",
          "Protein":"g",
          "Carbohydrate":"g",
          "Total fat":"g",
          "Cholesterol":"mg",
          "Fiber":"g",
          "Water":"g",
          "Alcohol":"g",
          "Vitamin C":"mg"}
DataframeAddCol(df_food,dictcol)


Now📝, we have our new dataframe with numerical columns from originals.

In [ ]:
print(df_food.info())

New Dataframe with new numerical columns.

In [ ]:

df_food_No = df_food[['FDC_ID','Item','Category','Calories_kcal',
              'Protein_g',
              'Carbohydrate_g',
              'Total fat_g',
              'Cholesterol_mg',
              'Fiber_g',
              'Water_g',
              'Alcohol_g',
              'Vitamin C_mg']]


df_food_No.info()
               


what is the highest vitamin C food and it's other properties?⁉️

In [ ]:
highCfood = df_food_No[df_food['Vitamin C_mg'] == df_food['Vitamin C_mg'].max()]



In [ ]:
highCfood

Which fruit has the highest vitamin C content? 
🥗🥝🍊🍏🍓🍊
What are some other sources of vitamin C?
🍉🍈🍇🍅🍄🌭🌮🌯🌽🌾


In [ ]:
df_foodFruit = df_food[df_food['Category'].isin(['Fruits and Fruit Juices'])]
df_foodFruit_HvitC = df_foodFruit[df_foodFruit['Vitamin C_mg'] == df_foodFruit['Vitamin C_mg'].max()]
itemFruitHC = list(df_foodFruit_HvitC['Item'])
ConcVitC = df_foodFruit_HvitC['Vitamin C_mg']

print(f"Now,the fruit has the heighest vitamin C content,This fruit is named '{itemFruitHC[0].split(',')[0]}' and it has a concentration of vitamin C equal {int(ConcVitC)} mg.")


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
df_food_No['Vitamin_C_mg'] = df_food_No['Vitamin C_mg']
ConcVitC = float(ConcVitC)

df_food_No['point_type'] = ['Highest vitC Fruit "Acerola"' if VitC == ConcVitC else 'Others' for VitC in df_food_No.Vitamin_C_mg]
sns.scatterplot(x = 'Vitamin_C_mg',
                y = 'Water_g',
                hue = 'point_type',
                data = df_food_No)
plt.show()


Other food with high concentration of Vitamin C.

In [ ]:

def highconc(column1,colNam='Vitamin C_mg',df0=df_food):
    """
    column1 is category of item
    column2 is numeric to take max
    take the max vitamin c in each item in column 
    return all dataframe columns
    but only unique item of selected column
    """
    listcolumn1 = list(column1.unique())
    df = pd.DataFrame()
    for i in listcolumn1:
        df1 = df0[column1.isin([i])]
        
        df2 = df1[df1[colNam] == df1[colNam].max()]
        df = pd.concat([df2, df], ignore_index=True)
        
    return df 

In [ ]:

dfhighC = highconc(df_food_No['Category'],df0=df_food_No)


In [ ]:

dfhighC.drop(index=18,axis=0,inplace=True)


The heighest vitamin C foods Dataframe in food Database.

In [ ]:

dfhighC[['Item','Category','Vitamin_C_mg']].sort_values('Vitamin_C_mg', ascending=False)


In [ ]:
df_foodVeg = df_food[df_food['Category'].isin(['Vegetables and Vegetable Products'])]

In [ ]:
Top10Fruit = df_foodFruit.sort_values('Vitamin C_mg', ascending= False).head(10)
Top10Veg = df_foodVeg.sort_values('Vitamin C_mg', ascending= False).head(10)
Top10dfFood = df_food.sort_values('Vitamin C_mg', ascending=False).head(10)

Alternative10 = dfhighC.sort_values('Vitamin C_mg',ascending=False).head(10)

 



Top ten of high concentration of foods in unique Categories.

In [ ]:
Alternative10[['Item','Category','Vitamin C_mg']].reset_index(drop=True)

Top high concentrations of all data notice duplicates.😁

In [ ]:

Top10dfFood[['Item','Vitamin C_mg']].reset_index(drop=True)

top vegetables with high concentration of vitamin c.

In [ ]:
Top10Veg[['Item','Vitamin C_mg']].reset_index(drop=True)

top 10 fruit with high concentration of vitamin c.

In [ ]:

Top10Fruit[['Item','Vitamin C_mg']].reset_index(drop=True)

Negative relationship between the calories and water content of a food item.

In [ ]:

sns.regplot(y='Calories_kcal',
            x='Water_g',
            data=df_food,
            ci=None)

plt.show()

##  What are the possible drawbacks of a zero-carb diet? 

In [ ]:
ZeroCarb = df_food_No[df_food_No['Carbohydrate_g' ]== 0.0]
ZeroCarb.info()

In [ ]:
ZeroCarb[['Calories_kcal','Total fat_g', 'Cholesterol_mg']].mean()

In [ ]:
df_food_No[['Calories_kcal','Total fat_g', 'Cholesterol_mg']].mean()

## What could be the drawbacks of a very high-protein diet?

In [ ]:
Top_500_high_protein = df_food_No.sort_values('Protein_g',ascending=False).head(500)
Top_500_high_protein.sort_values('Cholesterol_mg',ascending=False)

In [ ]:
Top_500_high_protein[['Calories_kcal','Total fat_g', 'Cholesterol_mg']].mean()

### Descriptive cholesterol comparison
These selected food groups have different cholesterol levels in this dataset. This comparison cannot establish disease risk or clinical diet effects.

## According to the Cleveland Clinic website, a gram of fat has around 9 kilocalories, and a gram of protein and a gram of carbohydrate contain 4 kilocalories each. Fit a linear model to test whether these estimates agree with the data.

In [ ]:
NoZeroCal = df_food_No[df_food_No['Calories_kcal'] != 0.0]
NoZeroCal['Fat_g'] = NoZeroCal['Total fat_g']

In [ ]:
NoZeroCal.info()

Now,time to the model.
Fit a linear model to test whether these estimates agree with the data.

In [ ]:

from statsmodels.formula.api import ols
# Fit the model
mdl_calories_vs_P_F_C = ols('Calories_kcal ~ Protein_g + Fat_g + Carbohydrate_g + 0', data=NoZeroCal).fit()

# Create the explanatory data
explanatory_dataPFC = pd.DataFrame({'Protein_g': [1, 0, 0,0,10,5.88,200],
                                   'Fat_g': [0, 1, 0,0,20,13.24,100],
                                   'Carbohydrate_g': [0, 0, 1,0,15,41.18,100]})
# Predict 'Calories_kcal'
prediction_data = explanatory_dataPFC.assign(Calories_kcal=mdl_calories_vs_P_F_C.predict(explanatory_dataPFC))

# Calculate MSE and RSE
mse = mdl_calories_vs_P_F_C.mse_resid
rse = np.sqrt(mse)

In [ ]:
df_food_No.iloc[0][3:7]

In [ ]:
print(mdl_calories_vs_P_F_C.params)

In [ ]:
explanatory_dataPFC

In [ ]:
print(mdl_calories_vs_P_F_C.rsquared)
print(mdl_calories_vs_P_F_C.rsquared_adj)

## Analyze the errors of your linear model to see what could be the hidden sources of calories in food.

In [ ]:
print('MSE :', mse)
print('RSE :', rse)

In [ ]:
prediction_data

In [ ]:

plt.figure()
sns.regplot(y='Calories_kcal',
            x='Protein_g',
            data=NoZeroCal,
            ci=None,
            scatter_kws={'alpha': 0.5},
            color=sns.color_palette("deep")[0],
            label='Protein')
sns.regplot(y='Calories_kcal',
            x='Fat_g',
            data=NoZeroCal,
            ci=None,
            scatter_kws={'alpha': 0.5},
            color=sns.color_palette("deep")[1],
            label='Fat')
sns.regplot(y='Calories_kcal',
            x='Carbohydrate_g',
            data=NoZeroCal,
            ci=None,
            scatter_kws={'alpha': 0.5},
            color=sns.color_palette("deep")[2],
            label='Carbohydrate')

plt.xlabel('Grams')
plt.ylabel('Calories (kcal)')
plt.legend()
plt.show()

In [ ]:
NoZeroCal

In [ ]:

from statsmodels.formula.api import ols
# Fit the model
mdl_calories_vs_P_F_C_cat = ols('Calories_kcal ~ Protein_g + Fat_g + Carbohydrate_g + Category + Category : Protein_g + Category:Fat_g + Category : Carbohydrate_g + 0', data=NoZeroCal).fit()

In [ ]:
from itertools import product 

In [ ]:
catList = list(NoZeroCal['Category'].unique())
proteinList = np.arange(0,50,10)
fatList = np.arange(0,50,10)
carbohydrateList = np.arange(0,50,10)


In [ ]:
p = product(catList, proteinList, fatList, carbohydrateList)

In [ ]:

explanatoryDataPFCcat = pd.DataFrame(p,
    columns=["Category",
"Protein_g",
"Fat_g",
"Carbohydrate_g"])

In [ ]:
explanatoryDataPFCcat

In [ ]:
mse1 = mdl_calories_vs_P_F_C_cat.mse_resid
rse1 = np.sqrt(mse1)
print(mdl_calories_vs_P_F_C_cat.rsquared_adj)
print(mdl_calories_vs_P_F_C_cat.rsquared)
print("MSE:",mse1)
print("RSE:",rse1)

In [ ]:
predictionData = explanatoryDataPFCcat.assign(
    Calories_kcal = mdl_calories_vs_P_F_C_cat.predict(explanatoryDataPFCcat)
)

In [ ]:
predictionData

In [ ]:
predp = predictionData[predictionData['Protein_g'] == 10] 
predf = predictionData[predictionData['Fat_g'] == 10] 
predcarb = predictionData[predictionData['Carbohydrate_g'] == 10] 

In [ ]:
predpZeroF = predp[predp['Fat_g'] == 0]
predfZeroP = predf[predf['Protein_g'] == 0]
predcarbZeroP = predcarb[predcarb['Protein_g'] == 0]

In [ ]:
predpZeroFZeroCarb = predpZeroF[predpZeroF['Carbohydrate_g'] == 0]
predfZeroPZeroCarb = predfZeroP[predfZeroP['Carbohydrate_g'] == 0]
predcarbZeroPZeroF = predcarbZeroP[predcarbZeroP['Fat_g'] == 0]

In [ ]:
print("Calories mean in 10 gm protein:",predpZeroFZeroCarb["Calories_kcal"].mean())
print("Calories mean in 10 gm Fat:",predfZeroPZeroCarb["Calories_kcal"].mean())
print("Calories mean in 10 gm carbohydrates :",predcarbZeroPZeroF["Calories_kcal"].mean())

### the last model is better that the first and it doesn't work with very small amount of fat , carbohydrates and proteins. we find that 1gm of fat equal 9 kcal and 1 gm of protein and carbohydrate equal 4 kcal.

## for example as first row of dataframe NoZeroCal we predict the calories of first row as 300 
## we find around 7 calories as a difference 

## to see what could be the hidden sources of calories in food

## fit model to discover the hidden sources of calories

## may be from fiber or alcohol. so we use df_food_Nna to drop any missing data. So we fit another model to predict if alcohol and fiber produce calories.

In [ ]:
df_food_Nna_no = DataframeAddCol(df_food_Nna,dictcol)

In [ ]:
NoZeroCalNna = df_food_Nna_no[df_food_Nna_no['Calories_kcal'] != 0.0]

In [ ]:
NoZeroCalNna.info()

In [ ]:
mdl_calories_vs_All = ols('Calories_kcal ~ Protein_g + Fat_g + Carbohydrate_g + Alcohol_g + Fiber_g + Category + Category : Protein_g + Category:Fat_g + Category : Carbohydrate_g + Category : Alcohol_g + Category : Fiber_g + 0', data=NoZeroCal).fit()

In [ ]:
CatList = list(NoZeroCalNna['Category'].unique())
AlcList = np.arange(0,50,10)
FibList = np.arange(0,50,10)
proteinList = np.arange(0,50,10)
fatList = np.arange(0,50,10)
carbohydrateList = np.arange(0,50,10)


In [ ]:
p1 = product(CatList, AlcList, FibList, proteinList, fatList, carbohydrateList )

In [ ]:

explanatoryDataAll = pd.DataFrame(p1,
    columns=["Category",
"Alcohol_g",
"Fiber_g",
            "Protein_g",
            "Fat_g",
            "Carbohydrate_g"])

In [ ]:
mse2 = mdl_calories_vs_All.mse_resid
rse2 = np.sqrt(mse2)
print(mdl_calories_vs_All.rsquared_adj)
print(mdl_calories_vs_All.rsquared)
print("MSE:",mse2)
print("RSE:",rse2)

In [ ]:
predictionDataAll = explanatoryDataAll.assign(
    Calories_kcal = mdl_calories_vs_All.predict(explanatoryDataAll)
)

In [ ]:
predictionDataAll

In [ ]:

def dfonecolvalue(df, col1, val1, val2,li):
    """
    make one column with val1 and selected column of df in li list set their values to val2.
    df: dataframe of choice.
    col1: column that you want to set to the selected value.
    val1: value you want to select from col1.
    val2: value you want to select from all columns of the dataframe.
    li list of columns that set values to val2
    

    Returns a new dataframe named as df + col1 + val1.
    """
    new_df = df.copy()  # Create a copy of the original dataframe
    new_df = new_df[new_df[col1]==val1]
    for i in li:
        new_df = new_df[new_df[i]==val2]
        
    return new_df
   


In [ ]:

listAlc = ["Fiber_g","Protein_g","Fat_g","Carbohydrate_g"]
listFiber = ["Alcohol_g","Protein_g","Fat_g","Carbohydrate_g"]
 
   
df_10_Alc_predict = dfonecolvalue(predictionDataAll,"Alcohol_g",10,0,listAlc)
df_10_Fib_predict = dfonecolvalue(predictionDataAll,"Fiber_g",10,0,listFiber)

In [ ]:

df_10_Alc_predict["Calories_kcal"].mean()

In [ ]:
df_10_Fib_predict["Calories_kcal"].mean()

### last we see that alcohol can give us around 7 kcal, this is hidden source of calories.

## please, leave comment to help me to improve my skills 
### thanks 🙏 🌹❤️